[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus_optimization/04_optimization_landscapes_and_convexity/first_principles.ipynb)

# Topic 04: Optimization Landscapes and Convexity

## 1. First-Principles Intuition & Motivation

Topic 03 ended with an honest but unsatisfying guarantee: for any smooth, bounded-below loss, gradient descent drives $\lVert \nabla f(\mathbf{x}_k) \rVert_2$ to zero at rate $O(k^{-1/2})$. It says the optimizer stops moving. It does not say the stopping place is any good.

A vanishing gradient is compatible with a global minimum, a local minimum, a local maximum, a saddle, and an entire flat plateau. Distinguishing them requires the next order of information — curvature — and deciding whether *local* information can ever be *globally* conclusive requires a structural hypothesis on $f$. That hypothesis is **convexity**, and its payoff is a single sentence: for a convex function, $\nabla f(\mathbf{x}^\star) = 0$ certifies a global minimum. Everything else in convex optimization is engineering around that fact.

Deep learning does not get that fact. Neural losses are non-convex for structural reasons — swap two hidden units and you get an equally good but different parameter vector, so equally-good minimizers are disconnected in an obviously non-convex way. And yet training works. The reconciliation, developed below, is that high-dimensional landscapes are not "full of bad local minima" the way low-dimensional cartoons suggest: they are dominated by **saddle points**, because a local minimum requires all $d$ curvature eigenvalues to agree in sign, and agreement of $d$ signs is exponentially improbable.

### A picture worth carrying: three surfaces

| Surface | Hessian at the origin | Character of the origin |
|---|---|---|
| $f(x,y) = x^2+y^2$ | $\operatorname{diag}(2,2) \succ 0$ | strict local (and global) minimum — a bowl |
| $f(x,y) = x^2-y^2$ | $\operatorname{diag}(2,-2)$, indefinite | saddle — a mountain pass, downhill in $y$, uphill in $x$ |
| $f(x,y) = x^2$ | $\operatorname{diag}(2,0) \succeq 0$, singular | a *valley floor*: a whole line of minimizers, curvature test inconclusive along $y$ |

The third row is the one that dominates modern practice. Overparameterized models sit on high-dimensional analogues of that valley floor: the Hessian at a solution has a large null space, the minimizers form a connected manifold rather than a point, and questions like "which minimum did we find?" become questions about *where on the manifold* we landed and how curved the surroundings are.

### Why convexity is the dividing line

Local information — the value, gradient, and Hessian at one point — is intrinsically myopic. Convexity is exactly the assumption that removes the myopia, and it does so through one inequality: *the graph never dips below any of its tangent planes*. Consequences follow immediately and without further work:

- A stationary point is a global minimizer (Proof 3.3), so a local certificate becomes a global one.
- The set of minimizers is convex, so there are no separated "basins" to be trapped between.
- Sublevel sets $\{\mathbf{x} : f(\mathbf{x}) \le c\}$ are convex, so descent methods cannot be walled off from the optimum.
- Adding $\mu$-strong convexity makes the minimizer unique and, by Topic 03, makes gradient descent converge geometrically.

Every modeling decision that "keeps the loss convex" — linear models with convex losses, $\ell_1$/$\ell_2$ penalties, SVM hinge loss, log-likelihoods of exponential families — is buying these four bullets. Everything a deep network does forfeits them, and must earn back a weaker substitute empirically.

### Notation used throughout

- $f: \mathbb{R}^d \to \mathbb{R}$ — the objective, assumed differentiable (and $C^2$ where Hessians appear); $\operatorname{dom} f$ its domain, always taken open and convex in second-order statements.
- $\mathbf{x}_c$ — a critical (stationary) point: $\nabla f(\mathbf{x}_c) = 0$. $\mathbf{x}^\star$ — a global minimizer; $f^\star = f(\mathbf{x}^\star)$.
- $H = \nabla^2 f$ — the Hessian, symmetric by Clairaut's theorem; $\lambda_1 \le \cdots \le \lambda_d$ its eigenvalues with orthonormal eigenvectors $\mathbf{v}_i$.
- $A \succeq 0$ — positive semidefinite ($\mathbf{u}^\top A\mathbf{u} \ge 0$ for all $\mathbf{u}$); $A \succ 0$ — positive definite; $A$ *indefinite* if it has eigenvalues of both signs.
- $\alpha(\mathbf{x}_c)$ — the **index** of a critical point: the fraction (or count) of negative Hessian eigenvalues. Index $0$ is a minimum, index $d$ a maximum, anything in between a saddle.
- $\mathcal{S} = \{\mathbf{x} : f(\mathbf{x}) \le c\}$ — a sublevel set; $\operatorname{epi} f = \{(\mathbf{x}, t) : t \ge f(\mathbf{x})\}$ — the epigraph.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Convex set).** $C \subseteq \mathbb{R}^d$ is convex if for all $\mathbf{x},\mathbf{y} \in C$ and $\theta \in [0,1]$, $\theta\mathbf{x}+(1-\theta)\mathbf{y} \in C$: the entire segment joining any two points of $C$ stays in $C$.

**Definition 2.2 (Convex function).** $f: C \to \mathbb{R}$ on a convex $C$ is convex if for all $\mathbf{x},\mathbf{y} \in C$ and $\theta\in[0,1]$,

$$
f\!\left(\theta\mathbf{x}+(1-\theta)\mathbf{y}\right) \le \theta f(\mathbf{x}) + (1-\theta)f(\mathbf{y}).
$$

It is *strictly* convex if the inequality is strict whenever $\mathbf{x} \neq \mathbf{y}$ and $\theta \in (0,1)$, and $\mu$-*strongly* convex if $f - \frac{\mu}{2}\lVert \cdot \rVert_2^2$ is convex. Equivalently, $f$ is convex iff $\operatorname{epi} f$ is a convex set.

**Theorem 2.3 (First-order characterization).** For differentiable $f$ on an open convex domain, $f$ is convex if and only if

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) \qquad \text{for all } \mathbf{x},\mathbf{y}.
$$

Geometrically: every tangent hyperplane is a global *underestimator* of $f$.

**Theorem 2.4 (Second-order characterization).** For $f \in C^2$ on an open convex domain, $f$ is convex if and only if $\nabla^2 f(\mathbf{x}) \succeq 0$ for all $\mathbf{x}$; and $f$ is $\mu$-strongly convex if and only if $\nabla^2 f(\mathbf{x}) \succeq \mu I$. ($\nabla^2 f \succ 0$ everywhere implies strict convexity, but not conversely — witness $f(x)=x^4$ at $x=0$.)

**Theorem 2.5 (Local implies global).** If $f$ is convex and $\mathbf{x}^\star$ is a local minimizer, then $\mathbf{x}^\star$ is a global minimizer. Moreover the set $\arg\min f$ is convex, and if $f$ is strictly convex it contains at most one point.

**Theorem 2.6 (Stationarity is sufficient under convexity).** If $f$ is convex and differentiable and $\nabla f(\mathbf{x}_c) = 0$, then $\mathbf{x}_c$ is a global minimizer.

**Definition 2.7 (Critical point taxonomy).** Let $\nabla f(\mathbf{x}_c)=0$ and $H = \nabla^2 f(\mathbf{x}_c)$ with eigenvalues $\lambda_1 \le \cdots \le \lambda_d$.

| Spectrum of $H$ | Name | Status |
|---|---|---|
| $\lambda_1 \gt 0$ ($H \succ 0$) | strict local minimum | sufficient second-order condition |
| $\lambda_d \lt 0$ ($H \prec 0$) | strict local maximum | sufficient |
| $\lambda_1 \lt 0 \lt \lambda_d$ | (strict) saddle point | sufficient: not a local extremum |
| $\lambda_1 = 0$ or $\lambda_d = 0$, no sign change | degenerate | inconclusive; higher order needed |

**Theorem 2.8 (Second-order optimality conditions).** Let $f \in C^2$ and $\mathbf{x}_c$ be a critical point.

1. *(Necessary.)* If $\mathbf{x}_c$ is a local minimizer, then $\nabla^2 f(\mathbf{x}_c) \succeq 0$.
2. *(Sufficient.)* If $\nabla^2 f(\mathbf{x}_c) \succ 0$, then $\mathbf{x}_c$ is a strict local minimizer.

The gap between $\succeq$ and $\succ$ is real: $f(x,y)=x^2-y^4$ has $\nabla^2 f(0)=\operatorname{diag}(2,0) \succeq 0$ at a point that is *not* a local minimum.

**Definition 2.9 (Strict saddle property).** $f$ satisfies the strict-saddle property if every critical point $\mathbf{x}_c$ either is a local minimizer or satisfies $\lambda_{\min}\!\left(\nabla^2 f(\mathbf{x}_c)\right) \lt 0$ — i.e. there are no degenerate saddles with only nonnegative curvature.

**Theorem 2.10 (Gradient descent escapes strict saddles; Lee et al., 2016).** If $f$ is $L$-smooth, $0 \lt \eta \lt 1/L$, and $f$ satisfies the strict-saddle property, then for Lebesgue-almost every initialization $\mathbf{x}_0$, gradient descent does not converge to a strict saddle. The set of initializations that do is a measure-zero stable manifold.

**Definition 2.11 (Sharpness).** Common scalar summaries of local flatness at a minimum $\mathbf{x}^\star$ are $\lambda_{\max}\!\left(\nabla^2 f(\mathbf{x}^\star)\right)$, the trace $\operatorname{tr}\nabla^2 f(\mathbf{x}^\star) = \sum_i\lambda_i$, and the $\epsilon$-sharpness $\max_{\lVert \boldsymbol{\delta} \rVert_2 \le \epsilon} f(\mathbf{x}^\star+\boldsymbol{\delta}) - f(\mathbf{x}^\star) \approx \frac{\epsilon^2}{2}\lambda_{\max}$. None is invariant under reparameterization (Dinh et al., 2017).

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: The first-order characterization of convexity

*Claim.* (Theorem 2.3) For differentiable $f$ on an open convex domain, $f$ is convex $\iff$ $f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x})$ for all $\mathbf{x},\mathbf{y}$.

*Proof.* **($\Rightarrow$) Convexity implies the tangent-plane bound.** Fix $\mathbf{x},\mathbf{y}$ and $\theta \in (0,1]$. Convexity applied to the point $\mathbf{x}+\theta(\mathbf{y}-\mathbf{x}) = \theta\mathbf{y}+(1-\theta)\mathbf{x}$ gives

$$
f\!\left(\mathbf{x}+\theta(\mathbf{y}-\mathbf{x})\right) \le (1-\theta)f(\mathbf{x}) + \theta f(\mathbf{y}).
$$

Subtract $f(\mathbf{x})$ from both sides and divide by $\theta \gt 0$:

$$
\frac{f\!\left(\mathbf{x}+\theta(\mathbf{y}-\mathbf{x})\right)-f(\mathbf{x})}{\theta} \le f(\mathbf{y}) - f(\mathbf{x}).
$$

Let $\theta \downarrow 0$. The left side is a difference quotient whose limit is the directional derivative $\nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x})$. Hence $\nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) \le f(\mathbf{y})-f(\mathbf{x})$, which rearranges to the claim.

**($\Leftarrow$) The tangent-plane bound implies convexity.** Let $\mathbf{z} = \theta\mathbf{x}+(1-\theta)\mathbf{y}$. Apply the assumed inequality twice, anchored at $\mathbf{z}$:

$$
f(\mathbf{x}) \ge f(\mathbf{z}) + \nabla f(\mathbf{z})^\top(\mathbf{x}-\mathbf{z}), \qquad f(\mathbf{y}) \ge f(\mathbf{z}) + \nabla f(\mathbf{z})^\top(\mathbf{y}-\mathbf{z}).
$$

Multiply the first by $\theta$, the second by $1-\theta$, and add. The gradient terms combine into $\nabla f(\mathbf{z})^\top\left(\theta\mathbf{x}+(1-\theta)\mathbf{y}-\mathbf{z}\right) = \nabla f(\mathbf{z})^\top \mathbf{0} = 0$, leaving

$$
\theta f(\mathbf{x}) + (1-\theta)f(\mathbf{y}) \ge f(\mathbf{z}) = f\!\left(\theta\mathbf{x}+(1-\theta)\mathbf{y}\right). \qquad \blacksquare
$$

*Contrast with Topic 02.* The descent lemma bounded $f$ *above* by a quadratic built from local data; convexity bounds $f$ *below* by a linear function built from local data. Optimization theory lives in the sandwich between them, and the width of that sandwich is $\kappa = L/\mu$.

### Proof 3.2: The second-order characterization $\nabla^2 f \succeq 0$

*Claim.* (Theorem 2.4) For $f \in C^2$ on an open convex domain, $f$ is convex $\iff$ $\nabla^2 f(\mathbf{x}) \succeq 0$ everywhere.

*Proof.* The whole argument is the 1D-restriction trick of Topic 02: define, for fixed $\mathbf{x}$ and direction $\mathbf{u}$, the scalar function

$$
g(t) = f(\mathbf{x}+t\mathbf{u}), \qquad g''(t) = \mathbf{u}^\top\nabla^2 f(\mathbf{x}+t\mathbf{u})\,\mathbf{u}.
$$

By Definition 2.2, $f$ is convex on its domain **iff** $g$ is convex on its interval domain for every $\mathbf{x}$ and every $\mathbf{u}$ (the chord condition for $f$ along the segment $\mathbf{x}\mathbf{y}$ *is* the chord condition for $g$ with $\mathbf{u}=\mathbf{y}-\mathbf{x}$). So it suffices to prove the 1D statement: $g$ convex $\iff g'' \ge 0$.

**($\Leftarrow$) $g'' \ge 0$ implies convex.** By Taylor with Lagrange remainder (Topic 02, Proof 3.1) at $k=1$, for any $s,t$ there is $\xi$ between them with $g(t) = g(s)+g'(s)(t-s)+\frac{g''(\xi)}{2}(t-s)^2 \ge g(s)+g'(s)(t-s)$. This is the 1D first-order condition, which by Proof 3.1 is equivalent to convexity.

**($\Rightarrow$) Convex implies $g'' \ge 0$.** Suppose $g''(t_0) \lt 0$ for some $t_0$. By continuity $g'' \lt 0$ on a neighborhood, so $g'$ is strictly decreasing there. But the first-order condition (applied twice and added) forces $g'$ to be non-decreasing:

$$
g(t) \ge g(s)+g'(s)(t-s), \quad g(s) \ge g(t)+g'(t)(s-t) \ \Longrightarrow \ \left(g'(t)-g'(s)\right)(t-s) \ge 0,
$$

a contradiction. Hence $g'' \ge 0$ everywhere.

Translating back: $g''(t) = \mathbf{u}^\top\nabla^2 f(\cdot)\mathbf{u} \ge 0$ for every direction $\mathbf{u}$ and every base point is precisely $\nabla^2 f \succeq 0$. $\blacksquare$

*Strong convexity for free.* Replacing "$g'' \ge 0$" by "$g'' \ge \mu$" runs the identical argument and yields $\nabla^2 f \succeq \mu I \iff \mu$-strong convexity — the hypothesis that gave geometric convergence in Topic 03.

### Proof 3.3: For convex $f$, local minima are global and stationarity suffices

*Claim.* (Theorems 2.5 and 2.6) Let $f$ be convex. (a) Any local minimizer is a global minimizer. (b) $\arg\min f$ is a convex set. (c) If $f$ is differentiable and $\nabla f(\mathbf{x}_c)=0$, then $\mathbf{x}_c$ is a global minimizer. (d) If $f$ is strictly convex, the minimizer is unique.

*Proof.* **(a)** Let $\mathbf{x}^\star$ be a local minimizer: $f(\mathbf{x}^\star) \le f(\mathbf{z})$ for all $\mathbf{z}$ with $\lVert \mathbf{z}-\mathbf{x}^\star \rVert_2 \lt r$. Suppose some $\mathbf{y}$ has $f(\mathbf{y}) \lt f(\mathbf{x}^\star)$. Take $\theta \in (0,1)$ small enough that $\mathbf{z}_\theta = \mathbf{x}^\star+\theta(\mathbf{y}-\mathbf{x}^\star)$ lies inside the ball, i.e. $\theta \lt r/\lVert \mathbf{y}-\mathbf{x}^\star \rVert_2$. Convexity gives

$$
f(\mathbf{z}_\theta) \le (1-\theta)f(\mathbf{x}^\star)+\theta f(\mathbf{y}) \lt (1-\theta)f(\mathbf{x}^\star)+\theta f(\mathbf{x}^\star) = f(\mathbf{x}^\star),
$$

contradicting local minimality. So no such $\mathbf{y}$ exists and $\mathbf{x}^\star$ is global.

**(b)** Let $\mathbf{x},\mathbf{y}$ both attain the minimum value $f^\star$ and $\theta\in[0,1]$. Then $f(\theta\mathbf{x}+(1-\theta)\mathbf{y}) \le \theta f^\star+(1-\theta)f^\star = f^\star$, and it cannot be smaller than the minimum, so equality holds and the whole segment consists of minimizers.

**(c)** By the first-order characterization (Proof 3.1) with $\mathbf{x}=\mathbf{x}_c$: for every $\mathbf{y}$,

$$
f(\mathbf{y}) \ge f(\mathbf{x}_c) + \underbrace{\nabla f(\mathbf{x}_c)}_{=\,\mathbf{0}}{}^\top(\mathbf{y}-\mathbf{x}_c) = f(\mathbf{x}_c).
$$

Hence $\mathbf{x}_c$ minimizes $f$ globally — a one-line proof that is the entire reason convexity is prized.

**(d)** If $\mathbf{x}\neq\mathbf{y}$ both minimize, then by strict convexity $f\!\left(\frac{\mathbf{x}+\mathbf{y}}{2}\right) \lt \frac{1}{2}f^\star+\frac{1}{2}f^\star = f^\star$, contradicting minimality. $\blacksquare$

*The asymmetry to remember.* Under convexity, a *local, checkable* condition yields a *global, unverifiable-by-search* conclusion. Without convexity, no amount of local information can certify global optimality — which is why deep-learning claims are always empirical claims about the reachable region, never certificates.

### Proof 3.4: Second-order optimality conditions and the classification of critical points

*Claim.* (Theorem 2.8 and Definition 2.7) Let $f\in C^2$ and $\nabla f(\mathbf{x}_c)=0$, $H = \nabla^2 f(\mathbf{x}_c)$. (a) If $\mathbf{x}_c$ is a local min then $H \succeq 0$. (b) If $H \succ 0$ then $\mathbf{x}_c$ is a strict local min. (c) If $H$ has both a positive and a negative eigenvalue, $\mathbf{x}_c$ is a saddle: neither a local min nor a local max.

*Proof.* Because $\nabla f(\mathbf{x}_c)=0$, the second-order Taylor expansion (Topic 02, Theorem 2.3) reduces to a pure curvature statement:

$$
f(\mathbf{x}_c+\mathbf{h}) = f(\mathbf{x}_c) + \frac{1}{2}\mathbf{h}^\top H\mathbf{h} + o\!\left(\lVert \mathbf{h} \rVert_2^2\right).
$$

**(a)** Suppose $\mathbf{u}^\top H\mathbf{u} \lt 0$ for some unit $\mathbf{u}$. Put $\mathbf{h} = t\mathbf{u}$:

$$
f(\mathbf{x}_c+t\mathbf{u}) - f(\mathbf{x}_c) = \frac{t^2}{2}\mathbf{u}^\top H\mathbf{u} + o(t^2) = t^2\left(\frac{\mathbf{u}^\top H\mathbf{u}}{2} + o(1)\right).
$$

For $t$ small the bracket is negative, so $f$ strictly decreases along $\mathbf{u}$ and $\mathbf{x}_c$ is not a local min. Contrapositive: local min $\Rightarrow H\succeq 0$.

**(b)** If $H \succ 0$ with smallest eigenvalue $\lambda_1 \gt 0$, then $\mathbf{h}^\top H\mathbf{h} \ge \lambda_1\lVert \mathbf{h} \rVert_2^2$ (Rayleigh quotient), so

$$
f(\mathbf{x}_c+\mathbf{h})-f(\mathbf{x}_c) \ge \frac{\lambda_1}{2}\lVert \mathbf{h} \rVert_2^2 + o\!\left(\lVert \mathbf{h} \rVert_2^2\right) = \lVert \mathbf{h} \rVert_2^2\left(\frac{\lambda_1}{2}+o(1)\right) \gt 0
$$

for all sufficiently small $\mathbf{h}\neq\mathbf{0}$. Strict local minimum.

**(c)** Let $\mathbf{v}_+$ and $\mathbf{v}_-$ be unit eigenvectors with $H\mathbf{v}_\pm = \lambda_\pm\mathbf{v}_\pm$, $\lambda_+ \gt 0 \gt \lambda_-$. Along $\mathbf{v}_+$: $f(\mathbf{x}_c+t\mathbf{v}_+) - f(\mathbf{x}_c) = \frac{t^2}{2}\lambda_+ + o(t^2) \gt 0$ for small $t$. Along $\mathbf{v}_-$: the same computation gives a negative value. So every neighborhood of $\mathbf{x}_c$ contains points with larger and points with smaller $f$: neither extremum. $\blacksquare$

**Why the degenerate case is genuinely undecidable.** With a zero eigenvalue the quadratic term vanishes along that direction and the $o(\lVert \mathbf{h}\rVert^2)$ remainder decides. All three outcomes occur at the origin with the same Hessian $\operatorname{diag}(2,0)$:

$$
f_1 = x^2+y^4 \ (\text{min}), \qquad f_2 = x^2-y^4 \ (\text{saddle}), \qquad f_3 = x^2 \ (\text{non-strict min, a whole line}).
$$

**Canonical form.** In the eigenbasis, writing $\tilde{\mathbf{h}} = Q^\top\mathbf{h}$, the local model is a sum of decoupled parabolas $f(\mathbf{x}_c)+\frac{1}{2}\sum_i \lambda_i\tilde{h}_i^2$. A critical point is thus fully described by the *signature* of $H$ — how many $\lambda_i$ are positive, negative, and zero — and its **index** $\alpha$ (the number of negative eigenvalues) interpolates from minimum ($\alpha=0$) through saddles to maximum ($\alpha=d$).

### Proof 3.5: Why high-dimensional landscapes are saddle-dominated

*Claim.* Under a symmetric random model of curvature, the probability that a given critical point in $\mathbb{R}^d$ is a local minimum decays exponentially in $d$; consequently critical points are overwhelmingly saddles, and the few minima that exist cluster at low loss.

*Proof (toy model, exact).* Suppose the $d$ Hessian eigenvalues at a random critical point are independent and each is positive with probability $p \in (0,1)$ — a crude but instructive stand-in for a random symmetric matrix's spectrum. A local minimum requires *all* $d$ eigenvalues positive, hence

$$
\Pr[\text{local min}] = p^{\,d}, \qquad \Pr[\text{local max}] = (1-p)^{d}, \qquad \Pr[\text{saddle}] = 1 - p^{d} - (1-p)^{d}.
$$

With the symmetric choice $p=\frac{1}{2}$: minima and maxima each occur with probability $2^{-d}$. In $d=10$ that is $10^{-3}$; in $d=100$ it is $10^{-30}$; in $d=10^{6}$ it is beyond astronomically small. More sharply, the *index* $\alpha = \#\{i : \lambda_i \lt 0\}$ is Binomial$(d,1-p)$, hence concentrates: $\alpha/d \to 1-p$ with fluctuations $O(d^{-1/2})$. A typical critical point has an $\Theta(d)$-dimensional subspace of *descent* directions. $\blacksquare$

*What the real theory adds.* For a Gaussian random field (and, via the spin-glass analogy of Choromanska et al., 2015, approximately for deep networks), the eigenvalues follow Wigner's semicircle law shifted by the loss value $E$ at the critical point, and Bray & Dean's computation gives

$$
\Pr[\alpha = 0 \mid \text{loss } E] \sim e^{-c\,d\,\Phi(E)},
$$

where the exponent vanishes as $E$ approaches the global minimum. Two conclusions Dauphin et al. (2014) drew from this and confirmed empirically:

1. **Index correlates with loss.** High-loss critical points have high index (many escape directions); low-loss critical points have low index. There is an approximately monotone "index–energy" relation, so bad minima are rare precisely because being a minimum forces you to be near the bottom.
2. **The real obstruction is slowness, not entrapment.** Near a saddle the gradient is small in *most* directions, so gradient descent crawls on a plateau for a long time before the negative-curvature mode grows. The escape is guaranteed but slow: along a direction with curvature $-\lvert\lambda\rvert$, the deviation grows as $(1+\eta\lvert\lambda\rvert)^k$, so escaping takes $O\!\left(\frac{1}{\eta\lvert\lambda\rvert}\log\frac{1}{\text{initial deviation}}\right)$ steps — long when $\lvert\lambda\rvert$ is tiny.

*Algorithmic response.* Saddle-free Newton replaces $H$ by $\lvert H\rvert$ (absolute values of eigenvalues), turning the step into a *descent* step along negative-curvature directions instead of the ascent step that raw Newton would take; perturbed gradient descent injects noise to break the measure-zero symmetry; and plain SGD's minibatch noise plays the same role for free.

### Proof 3.6: Gradient descent escapes strict saddles (the linearized argument)

*Claim.* Near a strict saddle $\mathbf{x}_c$ with $\lambda_{\min}(H) = -\gamma \lt 0$, gradient descent with $0 \lt \eta \lt 1/L$ has an expanding direction, and the set of initializations converging to $\mathbf{x}_c$ has Lebesgue measure zero.

*Proof (local linearization).* Write $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}_c$. Since $\nabla f(\mathbf{x}_c)=0$, Taylor gives $\nabla f(\mathbf{x}_k) = H\mathbf{e}_k + o(\lVert \mathbf{e}_k \rVert_2)$, so to first order

$$
\mathbf{e}_{k+1} = \left(I-\eta H\right)\mathbf{e}_k + o\!\left(\lVert \mathbf{e}_k \rVert_2\right),
$$

exactly the recursion of Topic 03 — but now $H$ is *indefinite*. In the eigenbasis, mode $i$ carries the factor $1-\eta\lambda_i$, and two regimes appear:

$$
\lambda_i \gt 0 \ \Rightarrow \ \lvert 1-\eta\lambda_i \rvert \lt 1 \ (\text{contracting}), \qquad \lambda_i = -\gamma \lt 0 \ \Rightarrow \ 1+\eta\gamma \gt 1 \ (\text{expanding}).
$$

(The contraction claim uses $\eta \lt 1/L \le 1/\lambda_i$.) So the negative-curvature component grows geometrically, $\lvert \tilde{e}_{k,-} \rvert \approx (1+\eta\gamma)^k\lvert \tilde{e}_{0,-} \rvert$, and the iterate is pushed away from $\mathbf{x}_c$ unless that component is *exactly* zero.

**Measure-zero conclusion.** Convergence to $\mathbf{x}_c$ therefore requires $\mathbf{x}_0$ to lie on the *stable manifold* — the set of points whose entire trajectory has zero component along every expanding mode. The gradient-descent map $T(\mathbf{x}) = \mathbf{x}-\eta\nabla f(\mathbf{x})$ has $DT = I-\eta\nabla^2 f$, which for $\eta \lt 1/L$ is invertible, so $T$ is a local diffeomorphism; the Center-Stable Manifold Theorem then says the stable set is an embedded submanifold of dimension $d - \#\{\lambda_i \lt 0\} \lt d$, hence of Lebesgue measure zero. A randomly initialized run avoids it with probability $1$ (Lee et al., 2016). $\blacksquare$

*The practical caveat.* "Almost surely escapes" says nothing about *how long*. If $\gamma$ is small, $(1+\eta\gamma)^k$ takes many steps to grow, and empirically the long plateaus in training curves are exactly this. Noise helps: a perturbation of size $\rho$ along the escape direction cuts the escape time to $O\!\left(\frac{1}{\eta\gamma}\log\frac{1}{\rho}\right)$, which is the design principle behind perturbed GD and part of why SGD is more robust than full-batch GD on these landscapes.

## 4. Computational & Algorithmic Insights

### 4.1 Testing definiteness without eigendecomposition

Deciding "is $H \succ 0$?" is the computational core of this module, and there are three practical routes:

| Method | Cost | Notes |
|---|---|---|
| Cholesky attempt $H = LL^\top$ | $O(d^3/3)$, fastest exact test | Succeeds iff $H \succ 0$; failure point reveals a negative-curvature direction |
| Eigenvalues (symmetric QR / Lanczos) | $O(d^3)$ dense; $O(k)$ matvecs for extremes | Gives the full signature; Lanczos gets $\lambda_{\min},\lambda_{\max}$ from Hessian-vector products alone |
| Leading principal minors (Sylvester) | $O(d^3)$ | Elegant on paper for $d \le 3$, numerically poor at scale |

At deep-learning scale one never forms $H$. The Hessian-vector product $\nabla^2 f(\mathbf{x})\mathbf{v} = \nabla\!\left(\nabla f(\mathbf{x})^\top\mathbf{v}\right)$ costs one extra backward pass (Topic 01), and feeding it to Lanczos gives the extreme eigenvalues — enough to answer "are we at a minimum or a saddle?" and to measure sharpness $\lambda_{\max}$. Hutchinson's estimator $\operatorname{tr}H \approx \frac{1}{m}\sum_j \mathbf{z}_j^\top H\mathbf{z}_j$ with Rademacher $\mathbf{z}_j$ gives the trace at $m$ matvecs.

### 4.2 Recognizing convexity without computing a Hessian

Verifying $\nabla^2 f \succeq 0$ symbolically is hopeless for composite objectives. Practitioners instead build convexity compositionally (Boyd & Vandenberghe, Ch. 3.2) — the rule set behind *disciplined convex programming* and libraries like CVXPY:

- **Atoms are convex.** Affine functions, norms $\lVert \cdot \rVert$, $\max$, $e^{ax}$, $-\log x$, $x\log x$, $\operatorname{logsumexp}$, quadratic forms with PSD matrices.
- **Nonnegative combination.** $f_1,f_2$ convex, $\alpha,\beta \ge 0$ $\Rightarrow$ $\alpha f_1+\beta f_2$ convex. (Loss $+$ regularizer stays convex.)
- **Pointwise max/sup.** $\sup_{i} f_i$ of convex $f_i$ is convex — this is why the hinge loss $\max(0, 1-y\hat{y})$ and worst-case/robust objectives are convex.
- **Affine precomposition.** $f$ convex $\Rightarrow$ $\mathbf{x}\mapsto f(A\mathbf{x}+\mathbf{b})$ convex. This single rule makes every linear model with a convex loss convex in its weights.
- **Composition.** $g$ convex non-decreasing and $h$ convex $\Rightarrow$ $g\circ h$ convex. (Fails without monotonicity: $g(u)=u^2$ is convex, $h(x)=x^2-1$ is convex, but so is the composite — whereas $g(u)=-u$ breaks it.)

The rules also explain the boundary: a two-layer network is $\mathbf{w}^{(2)\top}\phi(W^{(1)}\mathbf{x})$, a product of two learned factors, and *products of variables are not covered by any of these rules* — that is precisely where deep learning leaves convex territory.

### 4.3 Visualizing landscapes honestly

A $d$-dimensional surface with $d \sim 10^8$ can only be seen through slices, and naive slices lie. The standard protocol (Li et al., 2018):

1. Pick two random direction vectors $\boldsymbol{\delta},\boldsymbol{\eta}$ in parameter space and plot $g(\alpha,\beta) = f(\mathbf{x}^\star + \alpha\boldsymbol{\delta}+\beta\boldsymbol{\eta})$.
2. **Filter-normalize**: rescale each filter (or neuron) of $\boldsymbol{\delta}$ to have the same norm as the corresponding filter of $\mathbf{x}^\star$. Without this step, a network with small weights looks artificially "sharp" purely because a fixed-size perturbation is relatively larger — the scale artifact that Dinh et al. exploited.
3. Interpret with care: a random 2D slice through a $10^8$-dimensional space almost surely misses all negative-curvature directions, so *slices systematically make saddles look like minima*.

Other honest probes: interpolating along the segment between two independently trained solutions (reveals barriers, or their absence after permutation alignment — mode connectivity), plotting the Hessian eigenvalue *density* via stochastic Lanczos quadrature (reveals the characteristic bulk near zero plus a few large outliers), and tracking $\lambda_{\max}$ during training (reveals the "edge of stability" regime where $\eta\lambda_{\max}$ hovers around $2$, the exact threshold of Topic 03).

### 4.4 What algorithms do about curvature

| Situation | Failure mode of plain GD | Remedy |
|---|---|---|
| Strict saddle, small $\lvert \lambda_{\min}\rvert$ | plateau; escape takes $(1+\eta\gamma)^{-1}$-slow growth | noise (SGD, perturbed GD), momentum, negative-curvature steps |
| Negative curvature and Newton's method | raw Newton step $-H^{-1}\nabla f$ moves *uphill* along negative modes | saddle-free Newton ($\lvert H\rvert^{-1}\nabla f$), trust regions, Levenberg–Marquardt damping |
| Degenerate flat manifold of minima | no unique answer; Hessian singular | implicit regularization decides *where* on the manifold you land; explicit regularization restores strict convexity |
| Sharp minimum, large $\lambda_{\max}$ | training instability at $\eta\lambda_{\max} \gt 2$ | lower $\eta$, or actively seek flat regions (SAM: minimize $\max_{\lVert \boldsymbol{\epsilon}\rVert \le \rho} f(\mathbf{x}+\boldsymbol{\epsilon})$) |
| Ill-conditioned bowl | zig-zag, $O(\kappa)$ iterations | preconditioning, normalization layers, Adam, momentum (Topic 03) |

## 5. Real-World Physics & AI/ML Applications

### 5.1 Physics: energy landscapes, spin glasses, and transition states

The word "landscape" is borrowed from physics, where $f$ is a potential energy over configuration space. Minima are stable states, saddles are **transition states**, and the index-1 saddle connecting two basins sets the activation energy of the transition — this is the entire content of transition-state theory in chemistry, with rate $\propto e^{-\Delta E/k_BT}$ where $\Delta E$ is the barrier height at the saddle. Protein folding, glass formation, and crystal nucleation are all "which minimum does the dynamics find, and how do the saddles gate the search?" questions.

The **spin glass** is the sharpest analogue. For a random Hamiltonian over $d$ spins, Bray & Dean computed the expected number of critical points of each index and found the index–energy relation used in Proof 3.5: high-energy critical points are saddles of high index, and minima concentrate in a narrow band just above the ground state. Choromanska et al. (2015) argued that a deep ReLU network's loss, under simplifying assumptions, matches a spherical spin-glass Hamiltonian — supplying the theoretical scaffolding for the empirical observation that different training runs reach *different but nearly equally good* solutions. Langevin dynamics closes the loop with Topic 03: SGD with noise temperature $T \propto \eta/B$ samples $e^{-f/T}$, so the optimizer preferentially occupies wide basins simply because wide basins have more volume.

### 5.2 ML: what the landscape story does and does not license

**What the landscape picture explains well.** Training a large network almost never fails by getting stuck in a high-loss local minimum; it fails by plateauing (saddles and flat regions), by exploding (curvature above the stability threshold), or by conditioning (Topic 03). Overparameterization is the most reliable fix, and the landscape reason is structural: as width grows, the set of parameters achieving zero training loss becomes a high-dimensional connected manifold, spurious minima become non-generic, and the local geometry acquires PL-like behavior that restores fast convergence. Mode connectivity — the discovery that two independently trained solutions can be joined by a *low-loss curve* (Garipov et al., 2018; Draxler et al., 2018), and even by a straight line after permutation alignment — is direct evidence that the low-loss region is one connected object rather than a scatter of isolated wells.

**What remains genuinely open.** The sharp-versus-flat generalization story is the most-cited and least-settled claim in the area. Keskar et al. (2017) documented that large-batch training finds sharper minima with worse test error; the mechanism is Topic 03's noise scale $\eta/B$, which shrinks with large $B$ and lets the optimizer settle into narrow basins. But Dinh et al. (2017) showed the measure is broken as stated: for a ReLU network, the rescaling $\left(\alpha W^{(1)}, \alpha^{-1}W^{(2)}\right)$ leaves the *function* — hence its generalization — completely unchanged while multiplying Hessian eigenvalues by factors of $\alpha^{\pm 2}$, so any function can be made arbitrarily "sharp" by relabeling coordinates. The live research program is therefore to find *reparameterization-invariant* notions of flatness (normalized or relative sharpness, PAC-Bayes flatness, information-geometric measures using the Fisher metric rather than the Euclidean one), and algorithms like SAM that optimize a worst-case-in-a-ball objective work in practice while the theory is still being written.

**The practical checklist.** Diagnose a stalled run by asking, in order: is the gradient norm small (stationary) or just noisy? Is $\lambda_{\min}$ negative (saddle — wait, add noise, or use momentum) or near zero (degenerate flat region — the model may be underparameterized or the data degenerate)? Is $\eta\lambda_{\max}$ near $2$ (stability edge)? Is $\kappa$ enormous (conditioning — normalize)? Each branch maps to a theorem in this module or the previous one. The legacy notebook [`../optimization_landscape.ipynb`](../optimization_landscape.ipynb) makes each of these visible on small examples.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Where |
|---|---|---|
| Convex sets, convex functions, first/second-order conditions, composition rules | Boyd & Vandenberghe, *Convex Optimization* | Ch. 2, Ch. 3.1–3.2 |
| Critical points, saddles, plateaus and cliffs in deep nets | Goodfellow, Bengio & Courville, *Deep Learning* | Ch. 4.3, Ch. 8.2 |
| Necessary/sufficient second-order optimality conditions | Nocedal & Wright, *Numerical Optimization* | Thm. 2.3–2.4 |
| Saddle dominance in high dimension; saddle-free Newton | Dauphin et al. (2014), NeurIPS | §2–4 |
| Random-field index–energy relation | Bray & Dean (2007); Choromanska et al. (2015), AISTATS | whole; §3 |
| Strict saddles, measure-zero stable sets | Lee, Simchowitz, Jordan & Recht (2016), COLT | Thm. 4.1 |
| Filter-normalized loss-surface visualization | Li, Xu, Taylor, Studer & Goldstein (2018), NeurIPS | §4–5 |
| Large batch, sharp minima, generalization gap | Keskar et al. (2017), ICLR | §2–4 |
| Reparameterization critique of sharpness | Dinh, Pascanu, Bengio & Bengio (2017), ICML | §4–5 |
| Mode connectivity between minima | Garipov et al. (2018); Draxler et al. (2018) | §3; §2 |

**Backward pointers**: the second-order Taylor expansion that powers every classification here is proved in [`../02_taylor_approximation_and_local_models/`](../02_taylor_approximation_and_local_models/); the convergence machinery whose destination this module identifies is in [`../03_gradient_descent_mechanics/`](../03_gradient_descent_mechanics/).

**Forward pointers**: constrained landscapes, Lagrange multipliers, KKT conditions and duality — the convex-analysis payoff of this geometry — are developed in [`../../optimization/`](../../optimization/). Runnable visualizations of convex, non-convex, and saddle surfaces live in [`../optimization_landscape.ipynb`](../optimization_landscape.ipynb).